In [16]:
import warnings
warnings.filterwarnings('ignore')

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Le fichier de 5 millions de lignes est facultatif et ignoré par Git.
# À défaut, le notebook travaille sur le jeu brut du module.
BIG_SALES_PATH = RAW_DIR / 'ventes_volumineux.csv'
SALES_PATH = BIG_SALES_PATH if BIG_SALES_PATH.exists() else RAW_DIR / 'ventes_brutes.csv'

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Source PySpark :', SALES_PATH.name)
print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    if path.is_file():
        print(' ', path.name)


def save_dataset(df, name):
    """Enregistre un DataFrame Pandas en Parquet, avec repli CSV."""
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Source PySpark : ventes_brutes.csv
Données disponibles :
  .gitkeep
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


---
## Atelier 7.1 — Session Spark et schéma explicite (40 min)

### Partie guidée

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

# Limiter Spark à deux cœurs évite de surcharger Docker Desktop sur un petit jeu de données.
spark = (SparkSession.builder
         .appName('PMD-M1')
         .master('local[2]')
         .config('spark.driver.memory', '2g')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)
print('Cœurs disponibles :', spark.sparkContext.defaultParallelism)

ModuleNotFoundError: No module named 'pyspark'

**`spark.sql.shuffle.partitions` à 200 par défaut** est un réglage pensé pour un cluster.
En local il crée 200 tâches minuscules dont l'orchestration coûte plus cher que le calcul.
C'est la première cause de la réputation de lenteur de Spark sur petits volumes.

In [18]:
sales_path = str(SALES_PATH)

# Inférence du schéma : la colonne prix_unitaire, contenant aussi des chaînes
# comme « 123,45 EUR », doit rester de type texte.
start = time.perf_counter()
sales_inferred = spark.read.csv(sales_path, header=True, inferSchema=True)
print(f'Inférence : {time.perf_counter() - start:.1f} s')
sales_inferred.printSchema()

NameError: name 'spark' is not defined

In [19]:
# Schéma explicite : les clés restent des chaînes, même si certaines sont numériques.
# prix_unitaire et date_commande sont du texte à cause de leurs formats hétérogènes.
SALES_SCHEMA = StructType([
    StructField('id_commande', StringType(), True),
    StructField('date_commande', StringType(), True),
    StructField('id_client', StringType(), True),
    StructField('id_produit', StringType(), True),
    StructField('id_magasin', StringType(), True),
    StructField('quantite', IntegerType(), True),
    StructField('prix_unitaire', StringType(), True),
    StructField('remise_pct', DoubleType(), True),
    StructField('canal', StringType(), True),
    StructField('statut', StringType(), True),
    StructField('ville_livraison', StringType(), True),
])

start = time.perf_counter()
sales_raw = spark.read.csv(sales_path, header=True, schema=SALES_SCHEMA)
print(f'Lecture avec schéma explicite : {time.perf_counter() - start:.3f} s')
sales_raw.printSchema()

NameError: name 'StructType' is not defined

L'écart est spectaculaire, et il a une explication simple : avec un schéma explicite,
**rien n'a encore été lu**. Spark s'est contenté d'enregistrer une intention.

In [20]:
# La preuve : count() est une action, elle déclenche le parcours réel du CSV.
start = time.perf_counter()
n_rows = sales_raw.count()
print(f'{n_rows:,} lignes comptées en {time.perf_counter() - start:.1f} s')

NameError: name 'sales_raw' is not defined

### Partie autonome

In [21]:
# Q1 — Schéma explicite des ventes brutes
sales_q1 = spark.read.csv(
    str(RAW_DIR / 'ventes_brutes.csv'),
    header=True,
    schema=SALES_SCHEMA,
)
sales_q1.printSchema()

print(
    'prix_unitaire est déclaré StringType : certaines valeurs contiennent une virgule '
    'décimale et le suffixe EUR. On les convertira explicitement après nettoyage.'
)

NameError: name 'spark' is not defined

---
## Atelier 7.2 — Rejouer le nettoyage en PySpark (55 min)

### Partie guidée : le dictionnaire de traduction

| Pandas | PySpark |
|---|---|
| `df[df.a > 5]` | `df.filter(F.col('a') > 5)` |
| `df['b'] = df['a'] * 2` | `df.withColumn('b', F.col('a') * 2)` |
| `df[['a','b']]` | `df.select('a', 'b')` |
| `df.rename(columns={'a':'c'})` | `df.withColumnRenamed('a', 'c')` |
| `df.groupby('a')['b'].sum()` | `df.groupBy('a').agg(F.sum('b'))` |
| `df.drop_duplicates()` | `df.dropDuplicates()` |
| `df.dropna(subset=['a'])` | `df.na.drop(subset=['a'])` |
| `df.fillna({'a': 0})` | `df.na.fill({'a': 0})` |
| `df.sort_values('a')` | `df.orderBy('a')` |
| `df.head(5)` | `df.show(5)` ou `df.limit(5).toPandas()` |

**La différence de fond n'est pas syntaxique.** En Pandas, chaque ligne s'exécute
immédiatement. En Spark, elle s'ajoute à un plan qui ne sera évalué qu'à la première action.

In [22]:
# Aperçu des transformations sans action coûteuse sur le fichier entier
sales_preview = (sales_raw
    .withColumn(
        'prix_numerique',
        F.regexp_replace(
            F.regexp_replace(F.trim(F.col('prix_unitaire')), r'(?i)\s*EUR$', ''),
            ',', '.',
        ).cast('double'),
    )
    .withColumn('canal_normalise', F.lower(F.trim(F.col('canal'))))
    .limit(5))

sales_preview.show(truncate=False)
sales_preview.printSchema()

NameError: name 'sales_raw' is not defined

In [10]:
# Le plan montre les transformations prévues avant l'action
sales_preview.explain(mode='formatted')

NameError: name 'sales_preview' is not defined

Repérez dans le plan la remontée des filtres au plus près de la lecture du fichier
(*predicate pushdown*) et la sélection des seules colonnes utiles (*column pruning*).
Vous n'avez pas demandé ces optimisations : Catalyst les a déduites de votre plan.
C'est la contrepartie de l'évaluation paresseuse.

### Partie autonome

In [23]:
# Q2 — Nettoyage équivalent à la séance 3
# Le prix et la date sont d'abord conservés sous forme de texte, puis convertis
# avec des règles explicites. Les DataFrames Spark sont immuables et évalués paresseusement.

def normalized_text(column_name):
    """Expression Spark pour trim, espaces répétées, casse et accents courants."""
    value = F.lower(F.regexp_replace(F.trim(F.col(column_name)), r'\s+', ' '))
    for accented, plain in [('é', 'e'), ('è', 'e'), ('ê', 'e'), ('à', 'a'),
                            ('â', 'a'), ('î', 'i'), ('ï', 'i'), ('ô', 'o'),
                            ('ö', 'o'), ('ù', 'u'), ('û', 'u'), ('ü', 'u'),
                            ('ç', 'c')]:
        value = F.regexp_replace(value, accented, plain)
    return value


def clean_sales_dataframe(df):
    """Nettoie les ventes sans rapatrier le jeu entier dans la mémoire Python."""
    price_text = F.regexp_replace(
        F.regexp_replace(F.trim(F.col('prix_unitaire')), r'(?i)\s*EUR$', ''),
        ',', '.',
    )
    parsed_date = F.coalesce(
        F.to_timestamp('date_commande', 'yyyy-MM-dd'),
        F.to_timestamp('date_commande', 'dd/MM/yyyy'),
        F.to_timestamp('date_commande', 'dd-MM-yyyy'),
        F.to_timestamp('date_commande', 'yyyy/MM/dd HH:mm'),
        F.to_timestamp('date_commande', 'dd MMM yyyy'),
    )

    cleaned = (df
        .withColumn('prix_unitaire', price_text.cast('double'))
        .withColumn('date_commande', parsed_date)
        .withColumn('canal', normalized_text('canal'))
        .withColumn('statut', normalized_text('statut'))
        .withColumn('ville_livraison', normalized_text('ville_livraison'))
        .dropDuplicates()
        .filter(F.col('date_commande').isNotNull())
        .filter(F.col('quantite') > 0)
        .na.fill({'remise_pct': 0.0})
        .withColumn(
            'montant',
            F.round(
                F.col('quantite') * F.col('prix_unitaire')
                * (1 - F.col('remise_pct') / 100),
                2,
            ),
        )
        .withColumn('jour', F.to_date('date_commande'))
        .withColumn('heure', F.hour('date_commande'))
    )
    return cleaned


clean_sales_spark = clean_sales_dataframe(sales_raw)
print('Plan construit : aucun calcul de nettoyage n’a encore été déclenché.')
clean_sales_spark.printSchema()

NameError: name 'sales_raw' is not defined

**Q3 — Réponse.** La conversion des dates est la partie la plus délicate : les valeurs
utilisent cinq formats, donc il faut les essayer explicitement avec coalesce(to_timestamp(...)).
Une conversion silencieuse avec un format unique transformerait une partie des dates en valeurs
nulles. En Spark, chaque transformation renvoie un nouveau DataFrame ; il faut donc chaîner les
opérations et déclencher une action (count, show ou une écriture) pour réellement les exécuter.

---
## Atelier 7.3 — Agrégations, jointures et partitions (45 min)

In [12]:
summary = (clean_sales_spark
    .groupBy('ville_livraison', 'canal')
    .agg(
        F.count('*').alias('nb_ventes'),
        F.round(F.sum('montant'), 2).alias('ca_net'),
        F.round(F.avg('montant'), 2).alias('panier_moyen'),
        F.expr('percentile_approx(montant, 0.5)').alias('montant_median_approche'),
        F.countDistinct('id_client').alias('clients_uniques'),
    )
    .orderBy(F.desc('ca_net')))

summary.show(10, truncate=False)

NameError: name 'clean_sales_spark' is not defined

`percentile_approx` calcule une médiane **approchée**. Une médiane exacte imposerait de
trier l'intégralité des données sur l'ensemble des machines, ce qui coûte infiniment plus
cher. Sur des volumes de cet ordre, l'approximation est indiscernable du résultat exact.

In [ ]:
# Écriture partitionnée : le canal facilite les lectures filtrées récurrentes.
output_path = str(PROCESSED_DIR / 'ventes_nettoyees_spark')

(clean_sales_spark
 .select('id_commande', 'date_commande', 'ville_livraison', 'canal',
         'montant', 'quantite')
 .write.mode('overwrite')
 .partitionBy('canal')
 .parquet(output_path))

print('Données Parquet partitionnées écrites dans', output_path)
for folder in sorted(os.listdir(output_path))[:6]:
    print(' ', folder)

Le partitionnement sur disque permet à une lecture filtrée sur `categorie` de ne toucher
que le sous-dossier concerné. Choisissez la clé de partitionnement en fonction des filtres
les plus fréquents, et évitez les clés de forte cardinalité : mille partitions de dix lignes
sont pires que pas de partitionnement du tout.

In [ ]:
# Q4 — Jointure ventes / produits / magasins et CA livré par région et catégorie
# Le même échantillon est utilisé dans Spark et Pandas pour une comparaison équitable.
SAMPLE_ROWS = 50_000
sales_sample_raw = sales_raw.limit(SAMPLE_ROWS)
sales_sample_clean = clean_sales_dataframe(sales_sample_raw)

products_spark = (spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(RAW_DIR / 'produits.csv')))
stores_spark = (spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(str(RAW_DIR / 'magasins.csv')))

sales_joined_spark = (sales_sample_clean
    .join(F.broadcast(products_spark), on='id_produit', how='left')
    .join(F.broadcast(stores_spark), on='id_magasin', how='left')
    .filter(F.col('statut') == 'livre')
    .withColumn('ca_realise', F.col('montant')))

spark_revenue = (sales_joined_spark
    .groupBy('region', 'categorie')
    .agg(
        F.count('*').alias('lignes_livrees'),
        F.round(F.sum('ca_realise'), 2).alias('ca_livre'),
    )
    .orderBy(F.desc('ca_livre')))

spark_revenue.show(50, truncate=False)

# Référence Pandas sur les mêmes premières lignes, avec le nettoyage de la séance 3.
import sys
sys.path.insert(0, str(ROOT / 'src'))
from cleaning import clean_sales

pandas_raw = pd.read_csv(SALES_PATH, nrows=SAMPLE_ROWS)
pandas_clean, pandas_log = clean_sales(pandas_raw, verbose=False)
pandas_products = pd.read_csv(RAW_DIR / 'produits.csv')
pandas_stores = pd.read_csv(RAW_DIR / 'magasins.csv')
pandas_joined = (pandas_clean
    .merge(pandas_products, on='id_produit', how='left', validate='many_to_one')
    .merge(pandas_stores, on='id_magasin', how='left', validate='many_to_one'))
pandas_joined = pandas_joined[pandas_joined['statut'].eq('livre')]
pandas_joined['ca_realise'] = pandas_joined['montant']
pandas_revenue = (pandas_joined
    .groupby(['region', 'categorie'], dropna=False)
    .agg(
        lignes_livrees=('id_commande', 'size'),
        ca_livre=('ca_realise', 'sum'),
    )
    .reset_index())
pandas_revenue['ca_livre'] = pandas_revenue['ca_livre'].round(2)

print('Résultat Pandas correspondant :')
display(pandas_revenue.sort_values('ca_livre', ascending=False).head(50))

In [ ]:
# Q5 — Le broadcast hint envoie les petits référentiels à chaque worker.
# Comme produits.csv et magasins.csv sont des dimensions minuscules,
# un BroadcastHashJoin évite de trier/shuffler toute la table de ventes.
sales_joined_spark.explain(mode='formatted')

---
## Atelier 7.4 — Où se situe le point de bascule (40 min)

C'est l'atelier dont vous devrez rendre compte en soutenance. La question *pourquoi avoir
utilisé Spark ici ?* appelle une réponse chiffrée, pas une intuition.

In [ ]:
def benchmark_pandas(path, n_rows):
    start = time.perf_counter()
    df = pd.read_csv(path, nrows=n_rows)
    price = pd.to_numeric(
        df['prix_unitaire'].astype('string').str.replace(r'(?i)\s*EUR$', '', regex=True)
          .str.replace(',', '.', regex=False).str.strip(),
        errors='coerce',
    )
    discount = pd.to_numeric(df['remise_pct'], errors='coerce').fillna(0)
    quantity = pd.to_numeric(df['quantite'], errors='coerce')
    df['montant'] = quantity * price * (1 - discount / 100)
    result = (df[df['montant'] > 0]
              .groupby(['ville_livraison', 'canal'], dropna=False)
              .agg(nb=('id_commande', 'count'), ca=('montant', 'sum')))
    duration = time.perf_counter() - start
    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    return duration, memory_mb, len(result)


def benchmark_spark(path, n_rows):
    start = time.perf_counter()
    df = spark.read.csv(path, header=True, schema=SALES_SCHEMA).limit(n_rows)
    price = F.regexp_replace(
        F.regexp_replace(F.trim(F.col('prix_unitaire')), r'(?i)\s*EUR$', ''),
        ',', '.',
    ).cast('double')
    discount = F.coalesce(F.col('remise_pct'), F.lit(0.0))
    quantity = F.col('quantite').cast('double')
    result = (df
        .withColumn('montant', quantity * price * (1 - discount / 100))
        .filter(F.col('montant') > 0)
        .groupBy('ville_livraison', 'canal')
        .agg(F.count('id_commande').alias('nb'), F.sum('montant').alias('ca')))
    n_groups = result.count()
    return time.perf_counter() - start, None, n_groups

In [ ]:
# Les volumes sont limités au nombre de lignes réellement présentes.
with open(SALES_PATH, encoding='utf-8') as source_file:
    available_rows = sum(1 for _ in source_file) - 1

SAMPLE_SIZES = sorted({
    min(size, available_rows)
    for size in [50_000, 250_000, 1_000_000, 3_000_000]
    if available_rows > 0
})
measurements = []

for n_rows in SAMPLE_SIZES:
    pandas_duration, memory_mb, pandas_groups = benchmark_pandas(SALES_PATH, n_rows)
    spark_duration, _, spark_groups = benchmark_spark(str(SALES_PATH), n_rows)
    measurements.append({
        'lignes': n_rows,
        'pandas_s': round(pandas_duration, 2),
        'spark_s': round(spark_duration, 2),
        'memoire_pandas_Mo': round(memory_mb, 1),
        'groupes_pandas': pandas_groups,
        'groupes_spark': spark_groups,
        'rapport_pandas_sur_spark': round(pandas_duration / spark_duration, 2),
    })

comparison = pd.DataFrame(measurements)
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(comparison['lignes'], comparison['pandas_s'], marker='o', label='pandas')
ax.plot(comparison['lignes'], comparison['spark_s'], marker='s', label='pyspark')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('nombre de lignes')
ax.set_ylabel('durée (s)')
ax.set_title('Point de bascule Pandas / PySpark')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interprétation attendue

Sur une machine unique, **Pandas gagne presque toujours en durée brute**. Spark paie un coût
fixe de démarrage et de coordination que le parallélisme local ne compense pas.

L'intérêt de Spark n'est donc pas la vitesse à cette échelle, mais :

- la capacité à traiter un volume qui ne tient pas en mémoire ;
- la possibilité de passer du local au cluster sans réécrire le code ;
- la lecture sélective de données partitionnées, sans charger le reste.

**Répondre « j'ai utilisé Spark parce que c'est plus rapide » en soutenance est une réponse
fausse et sera sanctionnée.** La bonne réponse porte sur le volume, la mémoire disponible
et la trajectoire du projet.

In [ ]:
# Q6 — Estimation, pas seuil universel.
# La mémoire par ligne vient des échantillons précédents. Une marge de 40 %
# tient compte des copies temporaires, jointures et allocations de l'interpréteur.
available_ram_mb = psutil.virtual_memory().available / 1024 ** 2
largest_sample = comparison.iloc[-1]
bytes_per_row = largest_sample['memoire_pandas_Mo'] * 1024 ** 2 / largest_sample['lignes']
safe_rows_estimate = int(available_ram_mb * 0.60 * 1024 ** 2 / bytes_per_row)

print(f'Mémoire disponible au moment de la mesure : {available_ram_mb:,.0f} Mo')
print(f'Empreinte Pandas observée : {bytes_per_row:,.0f} octets par ligne')
print(f'Estimation prudente du seuil : environ {safe_rows_estimate:,} lignes')
print(
    'Cette estimation dépend de la RAM disponible, des colonnes, des types et des copies '
    'intermédiaires. Elle ne remplace pas une mesure sur la machine cible.'
)

In [ ]:
spark.stop()
print('Session fermée.')

---
## Livrable de la séance

- un script `src/spark_pipeline.py` autonome, exécutable par `spark-submit`, produisant
  un Parquet partitionné ;
- le tableau comparatif chiffré Pandas / PySpark et le graphique associé ;
- un paragraphe écrit justifiant, pour votre projet, le choix de traiter telle étape en
  Spark plutôt qu'en Pandas.